In [ ]:
# ============================================
# Re-establish Snowflake Session
# ============================================
from snowflake.snowpark.context import get_active_session
session = get_active_session()

print("Session re-established!")
print("Session active: " + str(session is not None))

Getting Started with the EY Challenge
Welcome to the 2026 EY Data & AI Challenge!

Confirm setup of External Access Integration (EAI)

In [ ]:
%%sql -r dataframe_1
-- EY 2026 AI & Data Challenge
-- Snowflake Mandatory Setup Script

-- ------------------------------------------------------------
-- 1. Create Challenge Database
-- ------------------------------------------------------------
CREATE DATABASE IF NOT EXISTS EY_WATER_QUALITY;
USE DATABASE EY_WATER_QUALITY;

CREATE SCHEMA IF NOT EXISTS CHALLENGE;
USE SCHEMA CHALLENGE;

CREATE WAREHOUSE IF NOT EXISTS EY_WH
  WITH
    WAREHOUSE_SIZE = 'XSMALL'
    AUTO_SUSPEND = 120
    AUTO_RESUME = TRUE
    INITIALLY_SUSPENDED = TRUE;

USE WAREHOUSE EY_WH;

CREATE OR REPLACE NETWORK RULE EY_EXTERNAL_APIS_RULE
  MODE = EGRESS
  TYPE = HOST_PORT
  VALUE_LIST = (
    'earthengine.googleapis.com',
    'storage.googleapis.com',
    'oauth2.googleapis.com',
    'landsatlook.usgs.gov',
    'planetarycomputer.microsoft.com',
    'api.planetarycomputer.microsoft.com',
    'planetarycomputer.blob.core.windows.net',
    '*.blob.core.windows.net',
    '*.dfs.core.windows.net',
    'login.microsoftonline.com',
    'pypi.org',
    'pypi.python.org',
    'pythonhosted.org',
    'files.pythonhosted.org'
  );

CREATE OR REPLACE EXTERNAL ACCESS INTEGRATION EY_EXTERNAL_ACCESS
  ALLOWED_NETWORK_RULES = (EY_EXTERNAL_APIS_RULE)
  ENABLED = TRUE;

GRANT USAGE ON INTEGRATION EY_EXTERNAL_ACCESS TO ROLE PUBLIC;
GRANT USAGE ON DATABASE EY_WATER_QUALITY TO ROLE PUBLIC;
GRANT USAGE ON SCHEMA EY_WATER_QUALITY.CHALLENGE TO ROLE PUBLIC;
GRANT USAGE ON WAREHOUSE EY_WH TO ROLE PUBLIC;

SHOW INTEGRATIONS LIKE 'EY_EXTERNAL_ACCESS';
SHOW NETWORK RULES LIKE 'EY_EXTERNAL_APIS_RULE';
SHOW DATABASES LIKE 'EY_WATER_QUALITY';
SHOW WAREHOUSES LIKE 'EY_WH';

Import Python packages

In [ ]:
# ============================================
# Setup: Import Libraries and Create Session
# ============================================

# Suppress warnings
import warnings
warnings.filterwarnings('ignore')

# Core libraries
import os
from datetime import date
from tqdm import tqdm

# Data manipulation
import numpy as np
import pandas as pd
from IPython.display import display

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Spatial
from scipy.spatial import cKDTree

# ML - preprocessing
from sklearn.preprocessing import StandardScaler, RobustScaler
from sklearn.model_selection import train_test_split, KFold, cross_val_score

# ML - models
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor

# ML - metrics
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

# Planetary Computer / STAC (optional)
try:
    import pystac_client
    import planetary_computer as pc
    from odc.stac import stac_load
    from pystac.extensions.eo import EOExtension as eo
    print("✅ Planetary Computer libraries loaded!")
except Exception as e:
    print(f"⚠️ Planetary Computer not available: {e}")
    print("Continuing without them - not needed for model training!")

print("✅ All libraries imported successfully!")

Load Python Dependencies

In [ ]:
# ============================================
# Install Missing Packages
# ============================================
import subprocess
import sys

packages = [
    "pystac==1.11.0",
    "pystac_client==0.9.0", 
    "planetary_computer==1.0.0",
    "odc-stac==0.3.10",
    "rioxarray==0.17.0",
    "rasterio==1.4.3",
    "shapely==2.1.1",
    "tqdm==4.66.5",
    "xarray==2025.3.1",
    "geopandas==1.1.1",
    "netCDF4==1.7.2",
    "adlfs==2025.8.0",
    "zarr==2.17.2",
    "dask==2024.10.0",
    "numcodecs==0.12.1",
]

for pkg in packages:
    print(f"Installing {pkg}...")
    result = subprocess.run(
        [sys.executable, "-m", "pip", "install", pkg, "--quiet"],
        capture_output=True, text=True
    )
    if result.returncode == 0:
        print(f"  ✅ {pkg} installed")
    else:
        print(f"  ❌ {pkg} FAILED: {result.stderr[-200:]}")

print("\n✅ All installations attempted!")
print("⚠️  RESTART KERNEL NOW before running the next cell!")


Response Variable

In [ ]:
# ============================================
# Set Database and Schema Context
# ============================================
print("=" * 80)
print("🔧 SETTING SNOWFLAKE CONTEXT")
print("=" * 80)

# Set the database and schema
session.use_database("EY_WATER_QUALITY")
session.use_schema("CHALLENGE")

print("\n✅ Context set:")
print(f"   Database: EY_WATER_QUALITY")
print(f"   Schema: CHALLENGE")
print("=" * 80)

Response Variable

In [ ]:
# ============================================
# Load all training datasets from Snowflake tables
# ============================================
print("\n📊 Loading data from Snowflake tables...")

# Ensure we're in the correct database and schema
session.use_database("EY_WATER_QUALITY")
session.use_schema("CHALLENGE")

# Water quality training data (target variables)
Water_Quality_df = session.table("WATER_QUALITY_TRAINING").to_pandas()
print(f"✅ Water Quality Training: {Water_Quality_df.shape[0]:,} rows × {Water_Quality_df.shape[1]} columns")

# Landsat features (satellite data)
Landsat_Features_df = session.table("LANDSAT_FEATURES_TRAINING").to_pandas()
print(f"✅ Landsat Features Training: {Landsat_Features_df.shape[0]:,} rows × {Landsat_Features_df.shape[1]} columns")

# TerraClimate features (climate data)
TerraClimate_Features_df = session.table("TERRACLIMATE_FEATURES_TRAINING").to_pandas()
print(f"✅ TerraClimate Features Training: {TerraClimate_Features_df.shape[0]:,} rows × {TerraClimate_Features_df.shape[1]} columns")

# Validation datasets
Landsat_Features_val_df = session.table("LANDSAT_FEATURES_VALIDATION").to_pandas()
print(f"✅ Landsat Features Validation: {Landsat_Features_val_df.shape[0]:,} rows × {Landsat_Features_val_df.shape[1]} columns")

TerraClimate_Features_val_df = session.table("TERRACLIMATE_FEATURES_VALIDATION").to_pandas()
print(f"✅ TerraClimate Features Validation: {TerraClimate_Features_val_df.shape[0]:,} rows × {TerraClimate_Features_val_df.shape[1]} columns")

print("\n✨ All datasets loaded successfully!")

# ============================================
# TerraClimate Data - Type Conversion & Validation
# ============================================
print("=" * 80)
print("🌍 TERRACLIMATE DATA - TYPE CONVERSION & VALIDATION")
print("=" * 80)

# Reuse already loaded data from Cell 6
Terraclimate_df = TerraClimate_Features_df.copy()

print(f"\n✅ TerraClimate data: {Terraclimate_df.shape[0]:,} rows × {Terraclimate_df.shape[1]} columns")
print("\n📋 Columns:", Terraclimate_df.columns.tolist())

# Convert PET to numeric
if 'PET' in Terraclimate_df.columns:
    Terraclimate_df['PET'] = pd.to_numeric(Terraclimate_df['PET'], errors='coerce')
    print(f"\n  ✅ PET converted to float: {Terraclimate_df['PET'].dtype}")

# Convert any remaining object columns to numeric
for col in Terraclimate_df.select_dtypes(include=['object']).columns:
    if 'DATE' not in col.upper():
        Terraclimate_df[col] = pd.to_numeric(Terraclimate_df[col], errors='coerce')
        print(f"  ✅ {col} converted to numeric")

# Handle NaN values
nan_count = Terraclimate_df.isna().sum().sum()
if nan_count > 0:
    print(f"\n  ⚠️ Found {nan_count} NaN values - filling with median...")
    Terraclimate_df = Terraclimate_df.fillna(Terraclimate_df.median(numeric_only=True))
    print("  ✅ NaNs filled with median values")
else:
    print("\n  ✅ No NaN values found - data is clean!")

print("\n📊 TerraClimate Summary:")
display(Terraclimate_df.describe())

print("\n" + "=" * 80)
print("✅ TerraClimate data ready!")
print("=" * 80)

Predictor Variables

In [ ]:
# ============================================
# Feature Engineering - Additional Features
# ============================================
session.use_database("EY_WATER_QUALITY")
session.use_schema("CHALLENGE")

print("=" * 80)
print("🔧 FEATURE ENGINEERING - ADDING NEW FEATURES")
print("=" * 80)

# Rename for consistency
landsat_train_features = Landsat_Features_df.copy()

# -------------------------------------------------------
# 1. DERIVED SPECTRAL INDICES
# -------------------------------------------------------
print("\n📡 Computing Spectral Indices...")

# NDWI - Normalized Difference Water Index
landsat_train_features['NDWI'] = (
    (landsat_train_features['GREEN'] - landsat_train_features['NIR']) /
    (landsat_train_features['GREEN'] + landsat_train_features['NIR'])
)
print("  ✅ NDWI computed (GREEN - NIR) / (GREEN + NIR)")

# NIR_SWIR_RATIO - Turbidity/sediment indicator
landsat_train_features['NIR_SWIR_RATIO'] = (
    landsat_train_features['NIR'] / landsat_train_features['SWIR22'].replace(0, np.nan)
)
print("  ✅ NIR_SWIR_RATIO computed (NIR / SWIR22)")

# SWIR_RATIO - Surface moisture indicator
landsat_train_features['SWIR_RATIO'] = (
    landsat_train_features['SWIR16'] / landsat_train_features['SWIR22'].replace(0, np.nan)
)
print("  ✅ SWIR_RATIO computed (SWIR16 / SWIR22)")

# -------------------------------------------------------
# 2. TEMPORAL FEATURES
# -------------------------------------------------------
print("\n📅 Extracting Temporal Features...")

# Convert SAMPLE_DATE to datetime
landsat_train_features['SAMPLE_DATE'] = pd.to_datetime(
    landsat_train_features['SAMPLE_DATE'], dayfirst=True
)

# Extract month and year
landsat_train_features['MONTH'] = landsat_train_features['SAMPLE_DATE'].dt.month
print("  ✅ MONTH extracted (1-12)")

landsat_train_features['YEAR'] = landsat_train_features['SAMPLE_DATE'].dt.year
print("  ✅ YEAR extracted")

# Extract season (Southern Hemisphere)
def get_season(month):
    if month in [12, 1, 2]:
        return 1  # Summer
    elif month in [3, 4, 5]:
        return 2  # Autumn
    elif month in [6, 7, 8]:
        return 3  # Winter
    else:
        return 4  # Spring

landsat_train_features['SEASON'] = landsat_train_features['MONTH'].apply(get_season)
print("  ✅ SEASON extracted (1=Summer, 2=Autumn, 3=Winter, 4=Spring)")

# -------------------------------------------------------
# 3. SUMMARY
# -------------------------------------------------------
print("\n" + "=" * 80)
print("📋 UPDATED FEATURE SET")
print("=" * 80)
print(f"\nTotal columns: {landsat_train_features.shape[1]}")
print(f"Total rows: {landsat_train_features.shape[0]:,}")
print("\nAll columns:")
for i, col in enumerate(landsat_train_features.columns, 1):
    print(f"  {i}. {col}")

print("\nSample data (first 3 rows):")
display(landsat_train_features.head(3))

# Check for NaN values introduced
nan_counts = landsat_train_features[
    ['NDWI', 'NIR_SWIR_RATIO', 'SWIR_RATIO', 'MONTH', 'YEAR', 'SEASON']
].isna().sum()
print("\n⚠️  NaN check on new features:")
print(nan_counts)

print("\n✅ Feature engineering complete!")
print("=" * 80)

In [ ]:
# ============================================
# Data Type Conversion - All Features
# ============================================
session.use_database("EY_WATER_QUALITY")
session.use_schema("CHALLENGE")

print("=" * 80)
print("🔧 DATA TYPE CONVERSION - ALL FEATURES")
print("=" * 80)

# -------------------------------------------------------
# 1. Original spectral indices
# -------------------------------------------------------
print("\n📡 Converting original spectral indices...")

for col in ['NDMI', 'MNDWI']:
    if col in landsat_train_features.columns:
        landsat_train_features[col] = landsat_train_features[col].astype(float)
        print(f"  ✅ {col} → {landsat_train_features[col].dtype}")
    else:
        print(f"  ⚠️  {col} not found")

# -------------------------------------------------------
# 2. New spectral indices
# -------------------------------------------------------
print("\n📡 Converting new spectral indices...")

for col in ['NDWI', 'NIR_SWIR_RATIO', 'SWIR_RATIO']:
    if col in landsat_train_features.columns:
        landsat_train_features[col] = pd.to_numeric(
            landsat_train_features[col], errors='coerce'
        ).astype(float)
        print(f"  ✅ {col} → {landsat_train_features[col].dtype}")
    else:
        print(f"  ⚠️  {col} not found")

# -------------------------------------------------------
# 3. Temporal features
# -------------------------------------------------------
print("\n📅 Converting temporal features...")

for col in ['MONTH', 'YEAR', 'SEASON']:
    if col in landsat_train_features.columns:
        landsat_train_features[col] = landsat_train_features[col].astype(int)
        print(f"  ✅ {col} → {landsat_train_features[col].dtype}")
    else:
        print(f"  ⚠️  {col} not found")

# -------------------------------------------------------
# 4. Core spectral bands
# -------------------------------------------------------
print("\n🛰️  Converting core spectral bands...")

for col in ['NIR', 'GREEN', 'SWIR16', 'SWIR22']:
    if col in landsat_train_features.columns:
        landsat_train_features[col] = pd.to_numeric(
            landsat_train_features[col], errors='coerce'
        ).astype(float)
        print(f"  ✅ {col} → {landsat_train_features[col].dtype}")
    else:
        print(f"  ⚠️  {col} not found")

# -------------------------------------------------------
# 5. Handle any NaN values introduced by coercion
# -------------------------------------------------------
print("\n🔧 Checking for NaN values after conversion...")
nan_counts = landsat_train_features.isna().sum()
nan_cols = nan_counts[nan_counts > 0]

if len(nan_cols) > 0:
    print(f"  ⚠️  Found NaNs in: {nan_cols.to_dict()}")
    print("  🔄 Filling NaNs with column medians...")
    landsat_train_features = landsat_train_features.fillna(
        landsat_train_features.median(numeric_only=True)
    )
    print("  ✅ NaNs filled with median values")
else:
    print("  ✅ No NaN values found — data is clean!")

# -------------------------------------------------------
# 6. Summary
# -------------------------------------------------------
print("\n" + "=" * 80)
print("📋 ALL COLUMN DATA TYPES AFTER CONVERSION:")
print("=" * 80)
print(landsat_train_features.dtypes)

print("\n" + "=" * 80)
print("📊 FINAL DATASET INFO:")
print("=" * 80)
print(f"  Rows:    {landsat_train_features.shape[0]:,}")
print(f"  Columns: {landsat_train_features.shape[1]}")

print("\nSample data (first 5 rows):")
display(landsat_train_features.head(5))

print("\n" + "=" * 80)
print("✅ ALL DATA TYPE CONVERSIONS COMPLETE!")
print("=" * 80)

Loading Pre-Extracted TerraClimate Data

In [ ]:
# Reuse already loaded data from Cell 6
Terraclimate_df = TerraClimate_Features_df.copy()

# ============================================
# TerraClimate Feature Engineering
# ============================================
session.use_database("EY_WATER_QUALITY")
session.use_schema("CHALLENGE")

print("=" * 80)
print("TERRACLIMATE FEATURE ENGINEERING")
print("=" * 80)


# -------------------------------------------------------
# 1. Convert data types
# -------------------------------------------------------
print("\nConverting data types...")
Terraclimate_df['PET'] = pd.to_numeric(Terraclimate_df['PET'], errors='coerce')
Terraclimate_df['SAMPLE_DATE'] = pd.to_datetime(Terraclimate_df['SAMPLE_DATE'], dayfirst=True)
print("  PET converted to float")
print("  SAMPLE_DATE converted to datetime")

# -------------------------------------------------------
# 2. PET derived features
# -------------------------------------------------------
print("\nEngineering PET-derived features...")

Terraclimate_df['PET_SQUARED'] = Terraclimate_df['PET'] ** 2
print("  PET_SQUARED computed")

Terraclimate_df['PET_LOG'] = np.log1p(Terraclimate_df['PET'].clip(lower=0))
print("  PET_LOG computed")

pet_monthly_mean = Terraclimate_df.groupby(
    Terraclimate_df['SAMPLE_DATE'].dt.month
)['PET'].transform('mean')
Terraclimate_df['PET_MONTHLY_ANOMALY'] = Terraclimate_df['PET'] - pet_monthly_mean
print("  PET_MONTHLY_ANOMALY computed")

# -------------------------------------------------------
# 3. Location-based features
# -------------------------------------------------------
print("\nEngineering location-based features...")

Terraclimate_df['LOC_CLUSTER'] = pd.cut(
    Terraclimate_df['LATITUDE'],
    bins=5,
    labels=[1, 2, 3, 4, 5]
).astype(float)
print("  LOC_CLUSTER computed")

# -------------------------------------------------------
# 4. Cyclical temporal features
# -------------------------------------------------------
print("\nExtracting cyclical temporal features...")

Terraclimate_df['TC_MONTH'] = Terraclimate_df['SAMPLE_DATE'].dt.month
Terraclimate_df['TC_YEAR'] = Terraclimate_df['SAMPLE_DATE'].dt.year

Terraclimate_df['MONTH_SIN'] = np.sin(2 * np.pi * Terraclimate_df['TC_MONTH'] / 12)
Terraclimate_df['MONTH_COS'] = np.cos(2 * np.pi * Terraclimate_df['TC_MONTH'] / 12)
print("  MONTH_SIN computed")
print("  MONTH_COS computed")

# -------------------------------------------------------
# 5. Handle NaN values
# -------------------------------------------------------
nan_count = Terraclimate_df.isna().sum().sum()
if nan_count > 0:
    print(f"\n  Found {nan_count} NaN values - filling with median...")
    Terraclimate_df = Terraclimate_df.fillna(Terraclimate_df.median(numeric_only=True))
    print("  NaNs filled")
else:
    print("\n  No NaN values - data is clean!")

# -------------------------------------------------------
# 6. Summary
# -------------------------------------------------------
print("\n" + "=" * 80)
print("UPDATED TERRACLIMATE FEATURE SET:")
print("=" * 80)
print(f"\nShape: {Terraclimate_df.shape[0]:,} rows x {Terraclimate_df.shape[1]} columns")
print("\nAll columns:")
for i, col in enumerate(Terraclimate_df.columns, 1):
    print(f"  {i}. {col}")

print("\nSample data (first 5 rows):")
display(Terraclimate_df.head(5))

print("\nSummary statistics:")
display(Terraclimate_df.describe())

print("\n" + "=" * 80)
print("TerraClimate feature engineering complete!")
print("=" * 80)

# PET interaction features
Terraclimate_df['PET_NIR_INTERACTION'] = (
    Terraclimate_df['PET'] * landsat_train_features['NIR']
)
Terraclimate_df['PET_NDMI_INTERACTION'] = (
    Terraclimate_df['PET'] * landsat_train_features['NDMI']
)
print("  ✅ PET interaction features computed")

In [ ]:
# ============================================
# Pre-Join Inspection
# ============================================
session.use_database("EY_WATER_QUALITY")
session.use_schema("CHALLENGE")

print("=" * 80)
print("PRE-JOIN INSPECTION")
print("=" * 80)

# Check shapes
print("\nDataset shapes:")
print(f"  Water_Quality_df:      {Water_Quality_df.shape}")
print(f"  landsat_train_features: {landsat_train_features.shape}")
print(f"  Terraclimate_df:        {Terraclimate_df.shape}")

# Check columns
print("\nWater Quality columns:")
print(Water_Quality_df.columns.tolist())

print("\nLandsat columns:")
print(landsat_train_features.columns.tolist())

print("\nTerraClimate columns:")
print(Terraclimate_df.columns.tolist())

# Check for duplicate columns between datasets
landsat_cols = set(landsat_train_features.columns)
terra_cols = set(Terraclimate_df.columns)
wq_cols = set(Water_Quality_df.columns)

print("\nDuplicate columns between Landsat and TerraClimate:")
print(landsat_cols.intersection(terra_cols))

print("\nDuplicate columns between Landsat and Water Quality:")
print(landsat_cols.intersection(wq_cols))

print("\nDuplicate columns between TerraClimate and Water Quality:")
print(terra_cols.intersection(wq_cols))

# Check all rows match
print("\nRow count check:")
print(f"  All match: {Water_Quality_df.shape[0] == landsat_train_features.shape[0] == Terraclimate_df.shape[0]}")

Joining the Predictor Variables and Response Variables

In [ ]:
# ============================================
# Join Predictor and Response Variables
# ============================================
print("=" * 80)
print("🔀 JOINING ALL DATASETS")
print("=" * 80)

# Reset indices to ensure clean concat
Water_Quality_df = Water_Quality_df.reset_index(drop=True)
landsat_train_features = landsat_train_features.reset_index(drop=True)
Terraclimate_df = Terraclimate_df.reset_index(drop=True)

# Combine all three datasets horizontally
wq_data = pd.concat(
    [Water_Quality_df, landsat_train_features, Terraclimate_df], 
    axis=1
)

# Remove duplicate columns
wq_data = wq_data.loc[:, ~wq_data.columns.duplicated()]

print(f"✅ Combined dataset: {wq_data.shape[0]:,} rows × {wq_data.shape[1]} columns")
print("\n📋 All columns:")
for i, col in enumerate(wq_data.columns, 1):
    print(f"  {i}. {col}")

# Handle missing values
wq_data = wq_data.fillna(wq_data.median(numeric_only=True))
print(f"\n✅ Missing values handled")
print(f"✅ wq_data ready for feature selection!")
print("=" * 80)

# Add spatial features
wq_data['LATITUDE'] = Water_Quality_df['LATITUDE'].values
wq_data['LONGITUDE'] = Water_Quality_df['LONGITUDE'].values
print("✅ Spatial features added: LATITUDE, LONGITUDE")

wq_data['PET_NIR_INTERACTION'] = wq_data['PET'] * wq_data['NIR']
wq_data['PET_NDMI_INTERACTION'] = wq_data['PET'] * wq_data['NDMI']
print("✅ PET interaction features added: PET_NIR_INTERACTION, PET_NDMI_INTERACTION")

print(f"\n✅ Final wq_data shape: {wq_data.shape[0]:,} rows × {wq_data.shape[1]} columns")
print(f"✅ wq_data ready for feature selection!")
print("=" * 80)

In [ ]:
# ============================================
# Feature Selection
# ============================================
print("=" * 80)
print("🎯 FEATURE SELECTION")
print("=" * 80)

# Define required columns
required_cols = [
    'SWIR22', 'NDMI', 'MNDWI', 'PET',
    'TOTAL_ALKALINITY',
    'ELECTRICAL_CONDUCTANCE',
    'DISSOLVED_REACTIVE_PHOSPHORUS'
]

# Check which columns are available
available_cols = [col for col in required_cols if col in wq_data.columns]
missing_cols = [col for col in required_cols if col not in wq_data.columns]

if missing_cols:
    print(f"\n⚠️ Missing columns: {missing_cols}")
else:
    print("\n✅ All required columns found!")

# CORRECT - keep full wq_data, just verify columns exist
wq_model_data = wq_data[available_cols]
print(f"\n✅ Final dataset: {wq_data.shape[0]:,} rows × {wq_data.shape[1]} columns")

# Define predictor and target columns
predictor_cols = ['SWIR22', 'NDMI', 'MNDWI', 'PET']
target_cols = [
    'TOTAL_ALKALINITY',
    'ELECTRICAL_CONDUCTANCE',
    'DISSOLVED_REACTIVE_PHOSPHORUS'
]

print("\n🎯 PREDICTOR VARIABLES:")
print("-" * 40)
for i, col in enumerate(predictor_cols, 1):
    if col in wq_data.columns:
        print(f"  {i}. {col}: min={wq_data[col].min():.3f}, max={wq_data[col].max():.3f}, mean={wq_data[col].mean():.3f}")

print("\n🎯 TARGET VARIABLES:")
print("-" * 40)
for i, col in enumerate(target_cols, 1):
    if col in wq_data.columns:
        print(f"  {i}. {col}: min={wq_data[col].min():.3f}, max={wq_data[col].max():.3f}, mean={wq_data[col].mean():.3f}")

print("\nSample data (first 5 rows):")
display(wq_data.head())

print("\nSummary statistics:")
display(wq_data.describe())

print("\n" + "=" * 80)
print("✅ DATA IS READY FOR MODEL TRAINING!")
print(f"  Predictor features: {len(predictor_cols)}")
print(f"  Target variables:   {len(target_cols)}")
print(f"  Total rows:         {wq_data.shape[0]:,}")
print("=" * 80)

Helper Functions:

In [ ]:
# ============================================
# Helper Functions
# ============================================
print("=" * 80)
print("🔧 HELPER FUNCTIONS")
print("=" * 80)

def split_data(X, y, test_size=0.3, random_state=42):
    return train_test_split(X, y, test_size=test_size, random_state=random_state)

def scale_data(X_train, X_test):
    scaler = RobustScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled  = scaler.transform(X_test)
    return X_train_scaled, X_test_scaled, scaler

def train_model(X_train_scaled, y_train):
    model = RandomForestRegressor(
        n_estimators=500,
        max_depth=8,
        min_samples_split=30,
        min_samples_leaf=15,
        max_features=0.4,
        random_state=42,
        n_jobs=-1
    )
    model.fit(X_train_scaled, y_train)
    return model

def evaluate_model(model, X_scaled, y_true, dataset_name="Test"):
    y_pred = model.predict(X_scaled)
    r2     = r2_score(y_true, y_pred)
    rmse   = np.sqrt(mean_squared_error(y_true, y_pred))
    mae    = mean_absolute_error(y_true, y_pred)
    print(f"\n  {dataset_name} Evaluation:")
    print(f"    R² Score: {r2:.4f}  (explains {r2*100:.1f}% of variance)")
    print(f"    RMSE:     {rmse:.4f}")
    print(f"    MAE:      {mae:.4f}")
    return y_pred, r2, rmse

def cross_validate_model(X, y, param_name="Parameter"):
    model = RandomForestRegressor(
        n_estimators=300,
        max_depth=12,
        min_samples_split=20,
        min_samples_leaf=10,
        max_features=0.6,
        random_state=42,
        n_jobs=-1
    )
    scaler   = RobustScaler()
    X_scaled = scaler.fit_transform(X)
    cv_scores = cross_val_score(
        model, X_scaled, y, cv=5, scoring='r2', n_jobs=-1
    )
    print(f"\n  {param_name} - 5-Fold Cross Validation:")
    print(f"    CV R² scores: {[round(s, 4) for s in cv_scores]}")
    print(f"    Mean CV R²:   {cv_scores.mean():.4f}")
    print(f"    Std CV R²:    {cv_scores.std():.4f}")
    if cv_scores.std() < 0.05:
        print(f"    ✅ Stable model - low variance across folds!")
    else:
        print(f"    ⚠️  Warning - high variance across folds!")
    return cv_scores

def run_pipeline(X, y, param_name="Parameter"):
    print(f"\n{'=' * 60}")
    print(f"Training Model for {param_name}")
    print(f"{'=' * 60}")

    X_train, X_test, y_train, y_test = split_data(X, y)
    print(f"  Train: {X_train.shape[0]} samples | Test: {X_test.shape[0]} samples")

    X_train_scaled, X_test_scaled, scaler = scale_data(X_train, X_test)
    model = train_model(X_train_scaled, y_train)

    print("\n  📊 Training Performance:")
    _, r2_train, rmse_train = evaluate_model(model, X_train_scaled, y_train, "Train")

    print("\n  📊 Test Performance:")
    _, r2_test, rmse_test = evaluate_model(model, X_test_scaled, y_test, "Test")

    # Overfitting check
    gap = r2_train - r2_test
    if gap > 0.1:
        print(f"\n  ⚠️  Possible overfitting - R² gap: {gap:.4f}")
    else:
        print(f"\n  ✅ Model generalizes well - R² gap: {gap:.4f}")

    results = pd.DataFrame([{
        "Parameter": param_name,
        "R2_Train": r2_train,
        "RMSE_Train": rmse_train,
        "R2_Test": r2_test,
        "RMSE_Test": rmse_test
    }])

    return model, scaler, results

print("✅ Helper functions defined:")
print("  1. split_data()           - 70/30 train/test split")
print("  2. scale_data()           - RobustScaler")
print("  3. train_model()          - Tuned Random Forest")
print("  4. evaluate_model()       - R², RMSE and MAE")
print("  5. cross_validate_model() - 5-fold cross validation")
print("  6. run_pipeline()         - Complete training pipeline")
print("=" * 80)

Define Features & Train Models:

 Function 1: Train and Test Split
 Split_data() - Splits data into 70% train, 30% test

In [ ]:
# ============================================
# Train and Test Split
# ============================================
print("=" * 80)
print("📊 TRAIN AND TEST SPLIT")
print("=" * 80)

# Use the FULL feature set from Cell 14
feature_cols = [
    'NIR', 'GREEN', 'SWIR16', 'SWIR22',
    'NDMI', 'MNDWI', 'NDWI',
    'NIR_SWIR_RATIO', 'SWIR_RATIO',
    'PET', 'PET_SQUARED', 'PET_LOG',
    'PET_MONTHLY_ANOMALY', 'LOC_CLUSTER',
    'PET_NIR_INTERACTION', 'PET_NDMI_INTERACTION',
    'MONTH', 'YEAR', 'SEASON',
    'MONTH_SIN', 'MONTH_COS',
     'LATITUDE', 'LONGITUDE' 
]

# CORRECT - use full wq_data not the narrowed version
available_features = [col for col in feature_cols if col in wq_data.columns]
print(f"\n✅ Using {len(available_features)} features:")
print(available_features)

# Define X and targets
X     = wq_data[available_features]
y_ta  = wq_data['TOTAL_ALKALINITY']
y_ec  = wq_data['ELECTRICAL_CONDUCTANCE']
y_drp = wq_data['DISSOLVED_REACTIVE_PHOSPHORUS']

print(f"\n✅ Feature matrix X shape: {X.shape}")

print("\n⏳ Splitting data (70% train | 30% test)...")

# Split for each target
X_train_ta,  X_test_ta,  y_train_ta,  y_test_ta  = split_data(X, y_ta)
X_train_ec,  X_test_ec,  y_train_ec,  y_test_ec  = split_data(X, y_ec)
X_train_drp, X_test_drp, y_train_drp, y_test_drp = split_data(X, y_drp)

print("\n✅ Split results:")
print(f"  Total Alkalinity:")
print(f"    Train: {X_train_ta.shape[0]:,} | Test: {X_test_ta.shape[0]:,}")
print(f"  Electrical Conductance:")
print(f"    Train: {X_train_ec.shape[0]:,} | Test: {X_test_ec.shape[0]:,}")
print(f"  Dissolved Reactive Phosphorus:")
print(f"    Train: {X_train_drp.shape[0]:,} | Test: {X_test_drp.shape[0]:,}")

print("\n" + "=" * 80)
print("✅ Train/Test split complete!")
print("=" * 80)

 Function 2: Feature Scaling
 Scale_data() - Scales features using StandardScaler

In [ ]:
# ============================================
# Feature Scaling
# ============================================
print("=" * 80)
print("📊 FEATURE SCALING")
print("=" * 80)
print("""
TIP 4 - PREPROCESSING IMPROVEMENT:
Benchmark uses StandardScaler (mean=0, std=1).
We use RobustScaler which is better for environmental data
because it uses median and IQR instead of mean and std.
This makes it resistant to outliers such as flood events
or drought periods that cause extreme water quality readings.
""")

# Scale for Total Alkalinity
print("🔧 Scaling features for Total Alkalinity...")
X_train_ta_scaled,  X_test_ta_scaled,  scaler_ta  = scale_data(X_train_ta,  X_test_ta)
print("  ✅ Total Alkalinity features scaled!")

# Scale for Electrical Conductance
print("🔧 Scaling features for Electrical Conductance...")
X_train_ec_scaled,  X_test_ec_scaled,  scaler_ec  = scale_data(X_train_ec,  X_test_ec)
print("  ✅ Electrical Conductance features scaled!")

# Scale for Dissolved Reactive Phosphorus
print("🔧 Scaling features for Dissolved Reactive Phosphorus...")
X_train_drp_scaled, X_test_drp_scaled, scaler_drp = scale_data(X_train_drp, X_test_drp)
print("  ✅ Dissolved Reactive Phosphorus features scaled!")

print("\n📊 Verifying scaled data statistics...")
scaled_stats = pd.DataFrame(
    X_train_ta_scaled,
    columns=available_features
).describe().round(4)

# Print as text instead of display for Snowflake compatibility
print(scaled_stats.to_string())

print("\n" + "=" * 80)
print("✅ Feature scaling complete!")
print("  • All features scaled using RobustScaler")
print("  • fit_transform on train | transform only on test")
print("  • No data leakage from test set!")
print("=" * 80)

 Function 3 :Model Training
 Function 3: train_model() - Trains Random Forest Regressor

In [ ]:
# ============================================
# Cross Validation + Model Training
# ============================================
print("=" * 80)
print("🔍 ANTI-OVERFITTING CHECK + MODEL TRAINING")
print("=" * 80)

# -------------------------------------------------------
# Step 1: Cross Validation Before Training
# -------------------------------------------------------
print("\n📊 Running 5-fold cross validation...")
print("What to look for:")
print("  • Mean CV R² > 0.65  → good predictive power")
print("  • Std CV R²  < 0.05  → stable consistent model")
print("  • Train/Test gap < 0.10 → no overfitting")
print("-" * 60)

cv_ta  = cross_validate_model(X, y_ta,  "Total Alkalinity")
cv_ec  = cross_validate_model(X, y_ec,  "Electrical Conductance")
cv_drp = cross_validate_model(X, y_drp, "Dissolved Reactive Phosphorus")

print("\n📊 Cross Validation Summary:")
print(f"  Total Alkalinity         - Mean R²: {cv_ta.mean():.4f}  | Std: {cv_ta.std():.4f}")
print(f"  Electrical Conductance   - Mean R²: {cv_ec.mean():.4f}  | Std: {cv_ec.std():.4f}")
print(f"  Dissolved React. Phosph. - Mean R²: {cv_drp.mean():.4f} | Std: {cv_drp.std():.4f}")

# Overfitting warning
print("\n🔍 Stability Check:")
for name, cv in [("Total Alkalinity", cv_ta), 
                  ("Electrical Conductance", cv_ec),
                  ("Dissolved Reactive Phosphorus", cv_drp)]:
    if cv.mean() > 0.65 and cv.std() < 0.05:
        print(f"  ✅ {name} - Stable and predictive")
    elif cv.std() >= 0.05:
        print(f"  ⚠️  {name} - High variance, possible instability")
    else:
        print(f"  ⚠️  {name} - Low predictive power, consider more features")

# -------------------------------------------------------
# Step 2: Train Models
# -------------------------------------------------------
print("\n" + "=" * 80)
print("🤖 TRAINING MODELS")
print("=" * 80)

# Train Model 1: Total Alkalinity
print("\n🎯 Training Model 1: Total Alkalinity...")
model_ta = train_model(X_train_ta_scaled, y_train_ta)
print(f"  ✅ Trained | Trees: {model_ta.n_estimators} | Max depth: {model_ta.max_depth}")

# Train Model 2: Electrical Conductance
print("\n🎯 Training Model 2: Electrical Conductance...")
model_ec = train_model(X_train_ec_scaled, y_train_ec)
print(f"  ✅ Trained | Trees: {model_ec.n_estimators} | Max depth: {model_ec.max_depth}")

# Train Model 3: Dissolved Reactive Phosphorus
print("\n🎯 Training Model 3: Dissolved Reactive Phosphorus...")
model_drp = train_model(X_train_drp_scaled, y_train_drp)
print(f"  ✅ Trained | Trees: {model_drp.n_estimators} | Max depth: {model_drp.max_depth}")

print("\n" + "=" * 80)
print("✅ ALL MODELS TRAINED SUCCESSFULLY!")
print("  • model_ta  → Total Alkalinity")
print("  • model_ec  → Electrical Conductance")
print("  • model_drp → Dissolved Reactive Phosphorus")
print("=" * 80)

 Function 4: Model Evaluation
 Function 4: evaluate_model() - Evaluates model with R² and RMSE

In [ ]:
# ============================================
# Model Evaluation
# ============================================
session.use_database("EY_WATER_QUALITY")
session.use_schema("CHALLENGE")

print("=" * 80)
print("MODEL EVALUATION")
print("=" * 80)

print("""
Each model is evaluated using:
  R2 Score - Measures variance explained (higher is better)
  RMSE     - Average magnitude of errors (lower is better)
  MAE      - Mean absolute error (lower is better)

We evaluate on BOTH train and test sets.
A large gap between train and test R2 indicates overfitting.
Target: Train/Test gap < 0.10
""")

# -------------------------------------------------------
# Model 1: Total Alkalinity
# -------------------------------------------------------
print("=" * 80)
print("MODEL 1: TOTAL ALKALINITY (TA)")
print("=" * 80)
y_pred_train_ta, r2_train_ta, rmse_train_ta = evaluate_model(
    model_ta, X_train_ta_scaled, y_train_ta, "Train"
)
y_pred_ta, r2_ta, rmse_ta = evaluate_model(
    model_ta, X_test_ta_scaled, y_test_ta, "Test"
)
gap_ta = r2_train_ta - r2_ta
print(f"\n  Train/Test R2 gap: {gap_ta:.4f}", 
      "- Good!" if gap_ta < 0.10 else "- Warning: possible overfitting!")

# -------------------------------------------------------
# Model 2: Electrical Conductance
# -------------------------------------------------------
print("\n" + "=" * 80)
print("MODEL 2: ELECTRICAL CONDUCTANCE (EC)")
print("=" * 80)
y_pred_train_ec, r2_train_ec, rmse_train_ec = evaluate_model(
    model_ec, X_train_ec_scaled, y_train_ec, "Train"
)
y_pred_ec, r2_ec, rmse_ec = evaluate_model(
    model_ec, X_test_ec_scaled, y_test_ec, "Test"
)
gap_ec = r2_train_ec - r2_ec
print(f"\n  Train/Test R2 gap: {gap_ec:.4f}",
      "- Good!" if gap_ec < 0.10 else "- Warning: possible overfitting!")

# -------------------------------------------------------
# Model 3: Dissolved Reactive Phosphorus
# -------------------------------------------------------
print("\n" + "=" * 80)
print("MODEL 3: DISSOLVED REACTIVE PHOSPHORUS (DRP)")
print("=" * 80)
y_pred_train_drp, r2_train_drp, rmse_train_drp = evaluate_model(
    model_drp, X_train_drp_scaled, y_train_drp, "Train"
)
y_pred_drp, r2_drp, rmse_drp = evaluate_model(
    model_drp, X_test_drp_scaled, y_test_drp, "Test"
)
gap_drp = r2_train_drp - r2_drp
print(f"\n  Train/Test R2 gap: {gap_drp:.4f}",
      "- Good!" if gap_drp < 0.10 else "- Warning: possible overfitting!")

# -------------------------------------------------------
# Final Summary Table
# -------------------------------------------------------
print("\n" + "=" * 80)
print("FINAL MODEL PERFORMANCE SUMMARY")
print("=" * 80)

summary_df = pd.DataFrame({
    'Model': [
        'Total Alkalinity',
        'Electrical Conductance',
        'Dissolved Reactive Phosphorus'
    ],
    'R2 Train': [r2_train_ta,  r2_train_ec,  r2_train_drp],
    'R2 Test':  [r2_ta,        r2_ec,        r2_drp],
    'Gap':      [gap_ta,       gap_ec,       gap_drp],
    'RMSE':     [rmse_ta,      rmse_ec,      rmse_drp]
})
display(summary_df.round(4))

print("\nOverfitting Assessment:")
for _, row in summary_df.iterrows():
    status = "Good generalization!" if row['Gap'] < 0.10 else "Possible overfitting!"
    print(f"  {row['Model']}: gap={row['Gap']:.4f} - {status}")

print("\n" + "=" * 80)
print("MODEL EVALUATION COMPLETE!")
print("=" * 80)

Model Workflow (Pipeline)

Model Training and Evaluation for Each Parameter

Validation Data Preparation:

In [ ]:
# ============================================
# Validation Data Preparation
# ============================================
print("=" * 80)
print("📊 VALIDATION DATA PREPARATION")
print("=" * 80)

# Load validation datasets (already loaded in Cell 6)
# Reuse Landsat_Features_val_df and TerraClimate_Features_val_df

print("Loading validation data...")
landsat_val_features = Landsat_Features_val_df.copy()
Terraclimate_val_df  = TerraClimate_Features_val_df.copy()
test_file            = session.table("SUBMISSION_TEMPLATE").to_pandas()

print(f"  ✅ Submission template: {test_file.shape}")
print(f"  ✅ Landsat validation:  {landsat_val_features.shape}")
print(f"  ✅ TerraClimate val:    {Terraclimate_val_df.shape}")

# -------------------------------------------------------
# Apply SAME feature engineering as training data
# -------------------------------------------------------
print("\n🔧 Applying feature engineering to validation data...")

# Convert data types
for col in ['NIR','GREEN','SWIR16','SWIR22','NDMI','MNDWI']:
    if col in landsat_val_features.columns:
        landsat_val_features[col] = pd.to_numeric(landsat_val_features[col], errors='coerce')

# Spectral indices
landsat_val_features['NDWI'] = (
    (landsat_val_features['GREEN'] - landsat_val_features['NIR']) /
    (landsat_val_features['GREEN'] + landsat_val_features['NIR'])
)
landsat_val_features['NIR_SWIR_RATIO'] = (
    landsat_val_features['NIR'] / landsat_val_features['SWIR22'].replace(0, np.nan)
)
landsat_val_features['SWIR_RATIO'] = (
    landsat_val_features['SWIR16'] / landsat_val_features['SWIR22'].replace(0, np.nan)
)
print("  ✅ Spectral indices computed")

# Temporal features
landsat_val_features['SAMPLE_DATE'] = pd.to_datetime(
    landsat_val_features['SAMPLE_DATE'], dayfirst=True
)
landsat_val_features['MONTH']  = landsat_val_features['SAMPLE_DATE'].dt.month
landsat_val_features['YEAR']   = landsat_val_features['SAMPLE_DATE'].dt.year
landsat_val_features['SEASON'] = landsat_val_features['MONTH'].apply(get_season)
landsat_val_features['MONTH_SIN'] = np.sin(2 * np.pi * landsat_val_features['MONTH'] / 12)
landsat_val_features['MONTH_COS'] = np.cos(2 * np.pi * landsat_val_features['MONTH'] / 12)
print("  ✅ Temporal features computed")

# TerraClimate features
Terraclimate_val_df['PET'] = pd.to_numeric(Terraclimate_val_df['PET'], errors='coerce')
Terraclimate_val_df['SAMPLE_DATE'] = pd.to_datetime(
    Terraclimate_val_df['SAMPLE_DATE'], dayfirst=True
)
Terraclimate_val_df['PET_SQUARED'] = Terraclimate_val_df['PET'] ** 2
Terraclimate_val_df['PET_LOG']     = np.log1p(Terraclimate_val_df['PET'].clip(lower=0))
pet_val_monthly_mean = Terraclimate_val_df.groupby(
    Terraclimate_val_df['SAMPLE_DATE'].dt.month
)['PET'].transform('mean')
Terraclimate_val_df['PET_MONTHLY_ANOMALY'] = Terraclimate_val_df['PET'] - pet_val_monthly_mean
Terraclimate_val_df['LOC_CLUSTER'] = pd.cut(
    Terraclimate_val_df['LATITUDE'], bins=5, labels=[1,2,3,4,5]
).astype(float)
print("  ✅ TerraClimate features computed")

# -------------------------------------------------------
# Build validation feature matrix with SAME 23 features
# -------------------------------------------------------
print("\n🔧 Building validation feature matrix...")

val_data = pd.concat([
    landsat_val_features.reset_index(drop=True),
    Terraclimate_val_df.reset_index(drop=True)
], axis=1)
val_data = val_data.loc[:, ~val_data.columns.duplicated()]

# Add spatial features
val_data['LATITUDE']  = Terraclimate_val_df['LATITUDE'].values
val_data['LONGITUDE'] = Terraclimate_val_df['LONGITUDE'].values

# Add interaction features
val_data['PET_NIR_INTERACTION']  = val_data['PET'] * val_data['NIR']
val_data['PET_NDMI_INTERACTION'] = val_data['PET'] * val_data['NDMI']

# Select same 23 features used in training
submission_val_data = val_data[available_features].copy()
submission_val_data = submission_val_data.fillna(submission_val_data.median(numeric_only=True))

print(f"\n✅ Validation feature matrix: {submission_val_data.shape}")
print(f"✅ Features match training: {submission_val_data.shape[1] == len(available_features)}")
print(f"✅ Missing values: {submission_val_data.isna().sum().sum()}")

display(submission_val_data.head(3))
print("=" * 80)
print("✅ Validation data ready for predictions!")
print("=" * 80)

In [ ]:
# ============================================
# Generate Predictions and Submission CSV
# ============================================
print("=" * 80)
print("🚀 GENERATING SUBMISSION")
print("=" * 80)

# Generate predictions using 23 feature models
print("📊 Generating predictions...")
pred_TA  = model_ta.predict(scaler_ta.transform(submission_val_data))
pred_EC  = model_ec.predict(scaler_ec.transform(submission_val_data))
pred_DRP = model_drp.predict(scaler_drp.transform(submission_val_data))

print("  ✅ Total Alkalinity done!")
print("  ✅ Electrical Conductance done!")
print("  ✅ Dissolved Reactive Phosphorus done!")

# Prediction ranges
print("\n📊 Prediction ranges:")
print(f"  Total Alkalinity:              min={pred_TA.min():.2f}  | max={pred_TA.max():.2f}  | mean={pred_TA.mean():.2f}")
print(f"  Electrical Conductance:        min={pred_EC.min():.2f}  | max={pred_EC.max():.2f}  | mean={pred_EC.mean():.2f}")
print(f"  Dissolved Reactive Phosphorus: min={pred_DRP.min():.2f} | max={pred_DRP.max():.2f} | mean={pred_DRP.mean():.2f}")

# Check actual column names in test_file
print("test_file columns:", test_file.columns.tolist()) 

# Build submission dataframe

# Handle both uppercase (Snowflake) and mixed case column names
lon_col  = 'LONGITUDE'  if 'LONGITUDE'  in test_file.columns else 'Longitude'
lat_col  = 'LATITUDE'   if 'LATITUDE'   in test_file.columns else 'Latitude'
date_col = 'SAMPLE_DATE' if 'SAMPLE_DATE' in test_file.columns else 'Sample Date'
submission_df = pd.DataFrame({
    'Longitude':                     test_file[lon_col].values,
    'Latitude':                      test_file[lat_col].values,
    'Sample Date':                   test_file[date_col].values,
    'Total Alkalinity':              pred_TA,
    'Electrical Conductance':        pred_EC,
    'Dissolved Reactive Phosphorus': pred_DRP

})

print("\n📊 Submission preview:")
display(submission_df.head(10))
print(f"\nShape: {submission_df.shape}")

# Save to CSV
submission_df.to_csv("/tmp/submission.csv", index=False)
print("\n✅ Saved to /tmp/submission.csv!")

# Verify file
if os.path.exists("/tmp/submission.csv"):
    size = os.path.getsize("/tmp/submission.csv")
    print(f"✅ File size: {size:,} bytes")

# Upload to Snowflake sidebar
print("\n📤 Uploading to Snowflake sidebar...")
session.sql("""
    PUT file:///tmp/submission.csv
    snow://workspace/USER$.PUBLIC.DEFAULT$/versions/live/
    AUTO_COMPRESS=FALSE
    OVERWRITE=TRUE
""").collect()
print("✅ Uploaded successfully!")

print("\n" + "=" * 80)
print("✅ SUBMISSION COMPLETE!")
print("=" * 80)
print("  1. Look at LEFT sidebar in Snowflake")
print("  2. Click the Files tab")
print("  3. Find submission.csv")
print("  4. Click 3 dots next to it")
print("  5. Click Download")
print("  6. Upload to EY challenge platform")
print("=" * 80)

In [ ]:
# Alternative download method
session.sql("""
    GET snow://workspace/USER$.PUBLIC.DEFAULT$/versions/live/submission.csv
    file:///tmp/
""").collect()
print("✅ File available for download")
print("Check the Files tab in the left sidebar")

In [ ]:
# ============================================
# Save submission.csv to workspace sidebar
# ============================================
import pandas as pd
import os

# Save submission_df directly to workspace
submission_df.to_csv("/tmp/submission.csv", index=False)

# Upload to workspace files
session.sql("""
    PUT file:///tmp/submission.csv
    @~/submission.csv
    AUTO_COMPRESS=FALSE
    OVERWRITE=TRUE
""").collect()

print("✅ Done! Check sidebar for submission.csv")

In [ ]:
# Force save to workspace directory
submission_df.to_csv("/workspace/submission.csv", index=False)
print("✅ Saved to workspace!")

In [ ]:
# Check where files can be saved
import os

# Try different paths
paths_to_try = [
    "/workspace/",
    "/home/",
    "/tmp/",
    os.getcwd()
]

for path in paths_to_try:
    try:
        test_file_path = os.path.join(path, "test.txt")
        with open(test_file_path, "w") as f:
            f.write("test")
        print(f"✅ Writable: {path}")
        os.remove(test_file_path)
    except Exception as e:
        print(f"❌ Not writable: {path} - {e}")

print("\nCurrent directory:", os.getcwd())
print("Files in current dir:", os.listdir(os.getcwd()))

In [ ]:
# Save directly to current workspace directory
import os

output_path = os.path.join(os.getcwd(), "submission.csv")
submission_df.to_csv(output_path, index=False)

# Verify
if os.path.exists(output_path):
    size = os.path.getsize(output_path)
    print(f"✅ submission.csv saved to workspace!")
    print(f"✅ File size: {size:,} bytes")
    print(f"✅ Full path: {output_path}")
    print(f"\n🔄 Now REFRESH the left sidebar")
    print(f"   Right click in sidebar → Refresh")
    print(f"   submission.csv should appear!")
else:
    print("❌ File not saved - check error")